In [5]:
import pandas as pd
import numpy as np
import altair as alt

alt.data_transformers.disable_max_rows()

df = pd.read_csv('clinical_trials_cleaned.csv')

DEMO_COLS = ['black', 'white', 'asian', 'women', 'age_65']
DEMO_LABELS = {
    'black':  'Black / African American',
    'white':  'White',
    'asian':  'Asian',
    'women':  'Women',
    'age_65': 'Age 65+',
}
CAT_COLORS = {
    'Cancer':                '#D65F5F',
    'Cardiovascular':        '#4878CF',
    'Immune / Inflammatory': '#6ACC65',
    'Infectious Disease':    '#B47CC7',
    'Metabolic / Genetic':   '#F0A030',
    'Neurological':          '#8EBA42',
    'Other':                 '#988ED5',
}
CATEGORIES = sorted(df['disease_category'].unique().tolist())

In [6]:
# aggregate data for driver (one avg per category per demographic)
agg_rows = []
for col in DEMO_COLS:
    grp = df.groupby('disease_category')[col].mean().round(1).reset_index()
    grp.columns = ['disease_category', 'avg_value']
    grp['demographic'] = DEMO_LABELS[col]
    agg_rows.append(grp)
agg_df = pd.concat(agg_rows, ignore_index=True)

# individual drug data for driven (one row per drug per demographic)
drug_rows = []
for col in DEMO_COLS:
    sub = df[['brand_name', 'disease_category', 'disease', col]].copy()
    sub = sub.rename(columns={col: 'value'})
    sub['demographic'] = DEMO_LABELS[col]
    drug_rows.append(sub)
drug_df = pd.concat(drug_rows, ignore_index=True).dropna(subset=['value'])

print("Aggregated data shape:", agg_df.shape)
print("Individual drug data shape:", drug_df.shape)

Aggregated data shape: (35, 3)
Individual drug data shape: (759, 5)


In [7]:
# --- selections ---
demo_select = alt.selection_point(
    name='demo',
    fields=['demographic'],
    bind=alt.binding_select(
        options=[DEMO_LABELS[c] for c in DEMO_COLS],
        name='Demographic: '
    ),
    value=DEMO_LABELS['black'],
)

cat_select = alt.selection_point(
    name='cat',
    fields=['disease_category'],
    on='click',
    clear='dblclick',
)

color_scale = alt.Scale(
    domain=list(CAT_COLORS.keys()),
    range=list(CAT_COLORS.values()),
)

# --- driver: avg bar chart ---
driver = (
    alt.Chart(agg_df)
    .mark_bar()
    .encode(
        x=alt.X('disease_category:N',
                sort=CATEGORIES,
                title='Disease Category',
                axis=alt.Axis(labelAngle=-25, labelFontSize=11)),
        y=alt.Y('avg_value:Q',
                scale=alt.Scale(domain=[0, 100]),
                title='Average % of Trial Participants'),
        color=alt.condition(
            cat_select,
            alt.Color('disease_category:N',
                      scale=color_scale,
                      legend=alt.Legend(title='Disease Category')),
            alt.value('#cccccc'),
        ),
        tooltip=[
            alt.Tooltip('disease_category:N', title='Category'),
            alt.Tooltip('demographic:N',       title='Demographic'),
            alt.Tooltip('avg_value:Q',         title='Avg %', format='.1f'),
        ],
    )
    .transform_filter(demo_select)
    .add_params(cat_select, demo_select)
    .properties(
        width=620,
        height=280,
        title=alt.Title(
            'Average % of Trial Participants by Disease Category',
            subtitle='Click a bar to filter the histogram below. Double-click to reset.',
            fontSize=14,
            subtitleFontSize=11,
        ),
    )
)

# --- driven: histogram of individual drug values ---
driven = (
    alt.Chart(drug_df)
    .mark_bar(stroke='white', strokeWidth=0.5)
    .encode(
        x=alt.X('value:Q',
                bin=alt.Bin(maxbins=15),
                #scale=alt.Scale(domain=[0, 100]),
                title='% of Trial Participants'),
        y=alt.Y('count():Q', title='Number of Drugs'),
        color=alt.condition(
            cat_select,
            alt.Color('disease_category:N', scale=color_scale, legend=None),
            alt.value('#cccccc'),
        ),
        tooltip=[
            alt.Tooltip('disease_category:N', title='Category'),
            alt.Tooltip('value:Q',            title='%', format='.1f'),
            alt.Tooltip('count():Q',          title='Number of Drugs'),
        ],
    )
    .transform_filter(demo_select)
    .transform_filter(cat_select)
    .properties(
        width=620,
        height=260,
        title=alt.Title(
            'Distribution Across Individual Drugs',
            subtitle='Showing selected category. Double-click bar above to reset to all.',
            fontSize=14,
            subtitleFontSize=11,
        ),
    )
)

dashboard = alt.vconcat(driver, driven, spacing=30).configure_view(strokeWidth=0)
dashboard

alt.VConcatChart(...)

In [8]:
dashboard.save('interactive-dashboard.html')

In [9]:
cancer_df = (
    df[df['disease_category'] == 'Cancer']
    .dropna(subset=['black'])
    .sort_values('black')
)

extracredit1 = (
    alt.Chart(cancer_df)
    .mark_circle(size=80)
    .encode(
        x=alt.X('black:Q',
                scale=alt.Scale(domain=[0, 50]),
                title='% of Black Patients'),
        y=alt.Y('brand_name:N',
                sort=alt.EncodingSortField(field='black', order='ascending'),
                title='Drug',
                axis=alt.Axis(labelFontSize=9)),
        color=alt.value('#D65F5F'),
        tooltip=[
            alt.Tooltip('brand_name:N', title='Drug'),
            alt.Tooltip('disease:N',    title='Indication'),
            alt.Tooltip('black:Q',      title='Black %', format='.1f'),
            alt.Tooltip('white:Q',      title='White %', format='.1f'),
        ],
    )
    .properties(
        width=500,
        height=600,
        title=alt.Title(
            'Participation of Black Patients in Cancer Drug Trials',
            subtitle='Each dot is one approved cancer drug (2015–2018).',
            fontSize=14,
            subtitleFontSize=11,
        ),
    )
)
extracredit1

alt.Chart(...)

In [10]:
extracredit1.save('extracredit1.html')